In [1]:
# Install required packages if not already
!pip install geemap earthengine-api

import ee
import geemap
import os
import pandas as pd
from datetime import datetime, timedelta

# Authenticate & Initialize GEE
ee.Authenticate()
ee.Initialize()

# Define Shanghai bounding box
shanghai = ee.Geometry.Rectangle([120.85, 30.65, 122.15, 31.95])

# Generate monthly intervals
def get_monthly_intervals(start_year=2004, end_year=2024):
    start = datetime(start_year, 1, 1)
    end = datetime(end_year, 12, 31)
    intervals = []
    while start <= end:
        month_end = (start + timedelta(days=32)).replace(day=1) - timedelta(days=1)
        intervals.append((start.strftime('%Y-%m-%d'), month_end.strftime('%Y-%m-%d')))
        start = (start + timedelta(days=32)).replace(day=1)
    return intervals

monthly_periods = get_monthly_intervals()

# NDVI (MODIS)
def get_monthly_ndvi():
    ndvi = []
    modis = ee.ImageCollection("MODIS/006/MOD13Q1").select("NDVI").map(lambda img: img.divide(10000))
    for start, end in monthly_periods:
        monthly = modis.filterDate(start, end).mean().reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=shanghai,
            scale=250,
            maxPixels=1e13
        ).getInfo()
        ndvi.append(monthly.get('NDVI', None))
    return ndvi

# Rainfall (CHIRPS)
def get_monthly_rainfall():
    rainfall = []
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    for start, end in monthly_periods:
        monthly = chirps.filterDate(start, end).sum().reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=shanghai,
            scale=5000,
            maxPixels=1e13
        ).getInfo()
        rainfall.append(monthly.get('precipitation', None))
    return rainfall

# Temperature (ERA5)
def get_monthly_temperature():
    temp = []
    era5 = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY").select("temperature_2m")
    for start, end in monthly_periods:
        monthly = era5.filterDate(start, end).mean().reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=shanghai,
            scale=1000,
            maxPixels=1e13
        ).getInfo()
        temp.append(monthly.get('temperature_2m', None))
    return temp

# Population (WorldPop)
def get_monthly_population():
    pop = []
    for start, end in monthly_periods:
        year = datetime.strptime(start, "%Y-%m-%d").year
        try:
            pop_img = ee.Image(f"WorldPop/GP/100m/pop/{year}").select("population")
            mean_pop = pop_img.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=shanghai,
                scale=100,
                maxPixels=1e13
            ).getInfo()
            pop.append(mean_pop.get('population', None))
        except:
            pop.append(None)
    return pop

# Urban Expansion (MODIS)
def get_monthly_urban():
    urban = []
    modis = ee.ImageCollection("MODIS/006/MCD12Q1").select("LC_Type1")
    for start, end in monthly_periods:
        year = datetime.strptime(start, "%Y-%m-%d").year
        try:
            lc_img = modis.filterDate(f"{year}-01-01", f"{year}-12-31").first()
            urban_mask = lc_img.eq(13)
            urban_area = urban_mask.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=shanghai,
                scale=500,
                maxPixels=1e13
            ).getInfo()
            urban.append(urban_area.get('LC_Type1', None))
        except:
            urban.append(None)
    return urban

# Air Pollution (Sentinel-5P NO2)
def get_monthly_air_pollution():
    air = []
    s5p = ee.ImageCollection("COPERNICUS/S5P/NRTI/L3_NO2").select("NO2_column_number_density")
    for start, end in monthly_periods:
        monthly = s5p.filterDate(start, end).mean().reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=shanghai,
            scale=1000,
            maxPixels=1e13
        ).getInfo()
        air.append(monthly.get('NO2_column_number_density', None))
    return air

# Run data collection
print("⏳ Fetching monthly data for Shanghai...")

ndvi = get_monthly_ndvi()
rain = get_monthly_rainfall()
temp = get_monthly_temperature()
pop = get_monthly_population()
urban = get_monthly_urban()
air = get_monthly_air_pollution()

# Combine into DataFrame
df = pd.DataFrame({
    "Start_Date": [start for start, _ in monthly_periods],
    "NDVI": ndvi,
    "Rainfall": rain,
    "Temperature": temp,
    "Population": pop,
    "Urban": urban,
    "NO2": air
})

# Save to CSV
output_path = r"C:\maps\datasets\shanghai_monthly_2004_2024.csv"
df.to_csv(output_path, index=False)
print(f"✅ Dataset saved to: {output_path}")


Defaulting to user installation because normal site-packages is not writeable



Successfully saved authorization token.
⏳ Fetching monthly data for Shanghai...


C:\Users\irctc\AppData\Roaming\Python\Python312\site-packages\ee\deprecation.py:207: DeprecationWarning: 

Attention required for MODIS/006/MOD13Q1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MOD13Q1

  warnings.warn(warning, category=DeprecationWarning)


KeyboardInterrupt: 

In [2]:
import ee, pandas as pd, os
ee.Initialize()
out = "C:/maps/datasets/ndvi_shanghai_2004_2024.csv"
shanghai = ee.Geometry.Rectangle([120.85, 30.7, 122.2, 31.9])
years = list(range(2004, 2025))
modis = ee.ImageCollection('MODIS/061/MOD13Q1').select('NDVI')

def ndvi_year(y):
    img = modis.filterDate(f'{y}-01-01', f'{y+1}-01-01').mean()
    val = img.reduceRegion(ee.Reducer.mean(), shanghai, 250).get('NDVI')
    return ee.Feature(None, {'year': y, 'ndvi': val})

fc = ee.FeatureCollection([ndvi_year(y) for y in years])
vals = fc.reduceColumns(ee.Reducer.toList(2), ['year', 'ndvi']).getInfo()
df = pd.DataFrame({'Year': vals['list'][0], 'NDVI': [v/10000 if v else None for v in vals['list'][1]]})
os.makedirs(os.path.dirname(out), exist_ok=True)
df.to_csv(out, index=False)
print("✅ NDVI saved.")


✅ NDVI saved.


In [3]:
import ee
import pandas as pd
import os

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize()

# Define Shanghai bounding box
shanghai = ee.Geometry.Rectangle([120.85, 30.7, 122.2, 31.9])

# NDVI ImageCollection
ndvi_collection = ee.ImageCollection("MODIS/061/MOD13Q1").select("NDVI")

# Output path (set your desired path)
output_csv_path = "C:/maps/datasets/ndvi_shanghai_2004_2024.csv"

# List of years
years = list(range(2004, 2025))
ndvi_data = []

# Loop through each year and extract mean NDVI
for year in years:
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    yearly_ndvi = ndvi_collection.filterDate(start_date, end_date).mean()
    stats = yearly_ndvi.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=shanghai,
        scale=250,
        maxPixels=1e9
    )

    mean_ndvi = stats.get("NDVI").getInfo()
    if mean_ndvi is not None:
        ndvi_data.append((year, mean_ndvi / 10000))  # MODIS NDVI scale is *10000
    else:
        ndvi_data.append((year, None))

# Convert to DataFrame
df = pd.DataFrame(ndvi_data, columns=["Year", "NDVI"])

# Save to CSV
df.to_csv(output_csv_path, index=False)
print(f"✅ NDVI saved to: {output_csv_path}")


✅ NDVI saved to: C:/maps/datasets/ndvi_shanghai_2004_2024.csv


In [5]:
import ee
import pandas as pd

# Initialize Earth Engine
ee.Initialize()

# Define Shanghai geometry
shanghai = ee.Geometry.Polygon([
    [120.85, 30.68],
    [120.85, 31.90],
    [122.15, 31.90],
    [122.15, 30.68],
    [120.85, 30.68]
])

# Define years
years = list(range(2004, 2025))

# Function to calculate annual rainfall
def get_annual_rainfall(year):
    start_date = f'{year}-01-01'
    end_date = f'{year}-12-31'
    dataset = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterDate(start_date, end_date).filterBounds(shanghai)
    annual_rainfall = dataset.select('precipitation').sum()
    mean_rainfall = annual_rainfall.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=shanghai,
        scale=5000,
        maxPixels=1e13
    ).get('precipitation')
    return {'year': year, 'rainfall': mean_rainfall.getInfo()}

# Compute rainfall for each year
rainfall_data = [get_annual_rainfall(year) for year in years]

# Convert to DataFrame
df_rainfall = pd.DataFrame(rainfall_data)

# Save to CSV
output_path = 'C:/maps/datasets/shanghai_rainfall_2004_2024.csv'
df_rainfall.to_csv(output_path, index=False)
print(f'Rainfall data saved to {output_path}')



Rainfall data saved to C:/maps/datasets/shanghai_rainfall_2004_2024.csv


In [6]:
# Function to calculate annual temperature
def get_annual_temperature(year):
    start_date = f'{year}-01-01'
    end_date = f'{year}-12-31'
    dataset = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR').filterDate(start_date, end_date).filterBounds(shanghai)
    annual_temp = dataset.select('temperature_2m').mean()
    mean_temp = annual_temp.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=shanghai,
        scale=1000,
        maxPixels=1e13
    ).get('temperature_2m')
    return {'year': year, 'temperature': mean_temp.getInfo() - 273.15}  # Convert from Kelvin to Celsius

# Compute temperature for each year
temperature_data = [get_annual_temperature(year) for year in years]

# Convert to DataFrame
df_temperature = pd.DataFrame(temperature_data)

# Save to CSV
output_path = 'C:/maps/datasets/shanghai_temperature_2004_2024.csv'
df_temperature.to_csv(output_path, index=False)
print(f'Temperature data saved to {output_path}')


Temperature data saved to C:/maps/datasets/shanghai_temperature_2004_2024.csv


In [9]:
# Function to calculate annual urban fraction.
def get_annual_urban_fraction(year):
    dataset = ee.ImageCollection('MODIS/006/MCD12Q1') \
                .filter(ee.Filter.calendarRange(year, year, 'year')) \
                .first()
    urban = dataset.select('LC_Type1').eq(13)  # Class 13 corresponds to urban areas
    stats = urban.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=shanghai_geometry,
        scale=500,
        maxPixels=1e13
    )
    return {'year': year, 'urban_fraction': stats.get('LC_Type1').getInfo()}

# Compute urban fraction data for each year.
urban_data = [get_annual_urban_fraction(year) for year in years]

# Convert to DataFrame and save as CSV.
df_urban = pd.DataFrame(urban_data)
df_urban.to_csv('C:/maps/datasets/shanghai_urban_2004_2024.csv', index=False)
print("✅ Urban expansion data saved: C:/maps/datasets/shanghai_urban_2004_2024.csv")



C:\Users\irctc\AppData\Roaming\Python\Python312\site-packages\ee\deprecation.py:207: DeprecationWarning: 

Attention required for MODIS/006/MCD12Q1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MCD12Q1

  warnings.warn(warning, category=DeprecationWarning)


EEException: Image.select: Parameter 'input' is required and may not be null.

In [10]:
# Function to calculate urban area fraction
def get_urban_fraction(year):
    dataset = ee.ImageCollection('MODIS/006/MCD12Q1').filterDate(f'{year}-01-01', f'{year}-12-31').first()
    urban = dataset.select('LC_Type1').eq(13)  # Urban and Built-up class
    urban_area = urban.multiply(ee.Image.pixelArea())
    urban_stats = urban_area.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=shanghai,
        scale=500,
        maxPixels=1e13
    )
    total_urban_area = urban_stats.get('LC_Type1')
    return {'year': year, 'urban_area_sq_m': total_urban_area.getInfo()}

# Compute urban area fraction for each year
urban_data = [get_urban_fraction(year) for year in years]

# Convert to DataFrame
df_urban = pd.DataFrame(urban_data)

# Save to CSV
output_path = 'C:/maps/datasets/shanghai_urban_2004_2024.csv'
df_urban.to_csv(output_path, index=False)
print(f'Urban expansion data saved to {output_path}')


EEException: Image.select: Parameter 'input' is required and may not be null.

In [11]:
import ee
import pandas as pd

ee.Initialize()

shanghai_geometry = ee.Geometry.Polygon(
    [[[120.85, 30.68], [120.85, 31.90], [122.15, 31.90], [122.15, 30.68], [120.85, 30.68]]]
)

def get_population(year):
    image = ee.ImageCollection('CIESIN/GPWv411/GPW_Population_Count') \
        .filter(ee.Filter.date(f'{year}-01-01', f'{year}-12-31')) \
        .first()
    stats = image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=shanghai_geometry,
        scale=1000,
        maxPixels=1e13
    )
    return {'year': year, 'population': stats.get('population_count').getInfo()}

years = list(range(2004, 2025))
pop_data = [get_population(year) for year in years]
df_pop = pd.DataFrame(pop_data)
df_pop.to_csv('C:/maps/datasets/shanghai_population_2004_2024.csv', index=False)
print("✅ Population data saved.")


EEException: Image.reduceRegion: Parameter 'image' is required and may not be null.

In [12]:
import ee
import pandas as pd

ee.Initialize()

# Define Shanghai boundary
shanghai_geometry = ee.Geometry.Polygon(
    [[[120.85, 30.68], [120.85, 31.90], [122.15, 31.90], [122.15, 30.68], [120.85, 30.68]]]
)

# GPWv4 has data only for these years:
available_years = [2000, 2005, 2010, 2015, 2020]

def get_population(year):
    image = ee.Image(f'CIESIN/GPWv411/GPW_Population_Count/{year}')
    stats = image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=shanghai_geometry,
        scale=1000,
        maxPixels=1e13
    )
    return {'year': year, 'population': stats.get('population_count').getInfo()}

# Get available years only
pop_data = [get_population(year) for year in available_years]

# Save to CSV
df_pop = pd.DataFrame(pop_data)
df_pop.to_csv('C:/maps/datasets/shanghai_population_gpw_2000_2020.csv', index=False)
print("✅ Population data saved.")


EEException: Image.load: Image asset 'CIESIN/GPWv411/GPW_Population_Count/2000' not found (does not exist or caller does not have access).

In [13]:
import ee
import pandas as pd

ee.Initialize()

# Shanghai geometry
shanghai_geometry = ee.Geometry.Polygon(
    [[[120.85, 30.68], [120.85, 31.90], [122.15, 31.90], [122.15, 30.68], [120.85, 30.68]]]
)

# Available years in GPWv411
available_years = [2000, 2005, 2010, 2015, 2020]

def get_population(year):
    collection = ee.ImageCollection("CIESIN/GPWv411/GPW_Population_Count")
    image = collection.filter(ee.Filter.eq('system:index', str(year))).first()
    stats = image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=shanghai_geometry,
        scale=1000,
        maxPixels=1e13
    )
    return {'year': year, 'population': stats.get('population_count').getInfo()}

# Fetch population for each available year
pop_data = [get_population(year) for year in available_years]

# Save to CSV
df_pop = pd.DataFrame(pop_data)
df_pop.to_csv("C:/maps/datasets/shanghai_population_gpw_2000_2020.csv", index=False)
print("✅ Shanghai population data saved.")


EEException: Image.reduceRegion: Parameter 'image' is required and may not be null.

In [14]:
import ee
import pandas as pd

# Initialize Earth Engine
ee.Initialize()

# Define Shanghai boundary (or use a custom region)
shanghai = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level1") \
             .filter(ee.Filter.eq('ADM1_NAME', 'Shanghai')) \
             .geometry()

# Function to get population for a given year
def get_population(year):
    try:
        image = ee.Image(f'WorldPop/GP/100m/pop/{year}').clip(shanghai)
        stats = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=shanghai,
            scale=100,
            maxPixels=1e9
        )
        return {'year': year, 'population': stats.get('population').getInfo()}
    except Exception as e:
        print(f"Error for year {year}: {e}")
        return {'year': year, 'population': None}

# Fetch population from 2004 to 2020
years = list(range(2004, 2021))
population_data = [get_population(year) for year in years]

# Convert to DataFrame and save
df = pd.DataFrame(population_data)
df.to_csv('C:/maps/datasets/population_shanghai_2004_2020.csv', index=False)

print("Population data saved to C:/maps/datasets/population_shanghai_2004_2020.csv")


Error for year 2004: Image.load: Image asset 'WorldPop/GP/100m/pop/2004' not found (does not exist or caller does not have access).
Error for year 2005: Image.load: Image asset 'WorldPop/GP/100m/pop/2005' not found (does not exist or caller does not have access).
Error for year 2006: Image.load: Image asset 'WorldPop/GP/100m/pop/2006' not found (does not exist or caller does not have access).
Error for year 2007: Image.load: Image asset 'WorldPop/GP/100m/pop/2007' not found (does not exist or caller does not have access).
Error for year 2008: Image.load: Image asset 'WorldPop/GP/100m/pop/2008' not found (does not exist or caller does not have access).
Error for year 2009: Image.load: Image asset 'WorldPop/GP/100m/pop/2009' not found (does not exist or caller does not have access).
Error for year 2010: Image.load: Image asset 'WorldPop/GP/100m/pop/2010' not found (does not exist or caller does not have access).
Error for year 2011: Image.load: Image asset 'WorldPop/GP/100m/pop/2011' not

In [15]:
import ee
import pandas as pd

ee.Initialize()

# Define Shanghai boundary using a bounding box
shanghai_geometry = ee.Geometry.Rectangle([120.85, 30.68, 122.12, 31.88])  # Approx bounding box

# Years available in WorldPop/POP
available_years = list(range(2000, 2021))

def get_population(year):
    try:
        dataset_id = f'WorldPop/POP/{year}'
        image = ee.Image(dataset_id)
        population = image.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=shanghai_geometry,
            scale=100,
            maxPixels=1e12
        )
        total = population.get('population').getInfo()
        return {'year': year, 'population': total}
    except Exception as e:
        print(f"Error for year {year}: {e}")
        return {'year': year, 'population': None}

# Fetch population data
population_data = [get_population(year) for year in available_years]

# Save as CSV
df = pd.DataFrame(population_data)
df.to_csv('C:/maps/datasets/shanghai_population_2000_2020.csv', index=False)
print("Saved to C:/maps/datasets/shanghai_population_2000_2020.csv")


Error for year 2000: Image.load: Image asset 'WorldPop/POP/2000' not found (does not exist or caller does not have access).
Error for year 2001: Image.load: Image asset 'WorldPop/POP/2001' not found (does not exist or caller does not have access).
Error for year 2002: Image.load: Image asset 'WorldPop/POP/2002' not found (does not exist or caller does not have access).
Error for year 2003: Image.load: Image asset 'WorldPop/POP/2003' not found (does not exist or caller does not have access).
Error for year 2004: Image.load: Image asset 'WorldPop/POP/2004' not found (does not exist or caller does not have access).
Error for year 2005: Image.load: Image asset 'WorldPop/POP/2005' not found (does not exist or caller does not have access).
Error for year 2006: Image.load: Image asset 'WorldPop/POP/2006' not found (does not exist or caller does not have access).
Error for year 2007: Image.load: Image asset 'WorldPop/POP/2007' not found (does not exist or caller does not have access).
Error fo

In [16]:
import ee
import pandas as pd

ee.Initialize()

# Shanghai bounding box
shanghai_geometry = ee.Geometry.Rectangle([120.85, 30.68, 122.12, 31.88])

# Available years in this dataset
available_years = list(range(2000, 2021))

def get_population(year):
    try:
        # The correct dataset format
        dataset_id = f'WorldPop/GP/100m/pop/{year}'
        image = ee.Image(dataset_id)
        stats = image.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=shanghai_geometry,
            scale=100,
            maxPixels=1e12
        )
        population = stats.get('population').getInfo()
        return {'year': year, 'population': population}
    except Exception as e:
        print(f"Error for year {year}: {e}")
        return {'year': year, 'population': None}

# Get data
population_data = [get_population(year) for year in available_years]

# Save to CSV
df = pd.DataFrame(population_data)
df.to_csv('C:/maps/datasets/shanghai_population_2000_2020.csv', index=False)
print("Saved to C:/maps/datasets/shanghai_population_2000_2020.csv")


Error for year 2000: Image.load: Image asset 'WorldPop/GP/100m/pop/2000' not found (does not exist or caller does not have access).
Error for year 2001: Image.load: Image asset 'WorldPop/GP/100m/pop/2001' not found (does not exist or caller does not have access).
Error for year 2002: Image.load: Image asset 'WorldPop/GP/100m/pop/2002' not found (does not exist or caller does not have access).
Error for year 2003: Image.load: Image asset 'WorldPop/GP/100m/pop/2003' not found (does not exist or caller does not have access).
Error for year 2004: Image.load: Image asset 'WorldPop/GP/100m/pop/2004' not found (does not exist or caller does not have access).
Error for year 2005: Image.load: Image asset 'WorldPop/GP/100m/pop/2005' not found (does not exist or caller does not have access).
Error for year 2006: Image.load: Image asset 'WorldPop/GP/100m/pop/2006' not found (does not exist or caller does not have access).
Error for year 2007: Image.load: Image asset 'WorldPop/GP/100m/pop/2007' not

In [ ]:
import ee
import pandas as pd

ee.Initialize()


shanghai_geometry = ee.Geometry.Rectangle([120.85, 30.68, 122.12, 31.88])


available_years = [2000, 2005, 2010, 2015, 2020]

def get_population(year):
    dataset_id = f'WorldPop/GP/100m/pop/CHN_{year}'
    try:
        image = ee.Image(dataset_id)
        population = image.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=shanghai_geometry,
            scale=100,
            maxPixels=1e12
        )
        total = population.get('population').getInfo()
        return {'year': year, 'population': total}
    except Exception as e:
        print(f"Error for year {year}: {e}")
        return {'year': year, 'population': None}


population_data = [get_population(year) for year in available_years]


df = pd.DataFrame(population_data)
df.to_csv('C:/maps/datasets/shanghai_population.csv', index=False)
print("Saved to C:/maps/datasets/shanghai_population.csv")


Saved to C:/maps/datasets/shanghai_population.csv


In [18]:
import pandas as pd

# Manually enter available population data for Shanghai (example values, replace with actual)
data = {
    'year': [2000, 2005, 2010, 2015, 2020],
    'population': [17000000, 18500000, 20000000, 23000000, 24800000]  # Replace with actual values
}

df = pd.DataFrame(data)

# Interpolate yearly data between 2000 and 2020
df_full = df.set_index('year').reindex(range(2000, 2021)).interpolate(method='linear').reset_index()
df_full.columns = ['year', 'population']

# Save to CSV
df_full.to_csv('C:/maps/datasets/shanghai_population_2000_2020_interpolated.csv', index=False)
print("Interpolated yearly population saved.")


Interpolated yearly population saved.


In [19]:
import ee
import pandas as pd

ee.Initialize()

shanghai_geometry = ee.Geometry.Rectangle([120.85, 30.68, 122.12, 31.88])
years = list(range(2004, 2025))

def get_urban_expansion(year):
    try:
        image = ee.ImageCollection('MODIS/006/MOD44B') \
                    .filterDate(f'{year}-01-01', f'{year}-12-31') \
                    .select('Urban') \
                    .mean()
        stats = image.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=shanghai_geometry,
            scale=250,
            maxPixels=1e12
        )
        total_urban = stats.get('Urban').getInfo()
        return {'year': year, 'urban_area': total_urban}
    except Exception as e:
        print(f"Urban error {year}: {e}")
        return {'year': year, 'urban_area': None}

urban_data = [get_urban_expansion(year) for year in years]
pd.DataFrame(urban_data).to_csv('C:/maps/datasets/shanghai_urban_2004_2024.csv', index=False)


C:\Users\irctc\AppData\Roaming\Python\Python312\site-packages\ee\deprecation.py:207: DeprecationWarning: 

Attention required for MODIS/006/MOD44B! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MOD44B

  warnings.warn(warning, category=DeprecationWarning)


Urban error 2004: reduce.mean: Error in map(ID=2004_03_05):
Image.select: Band pattern 'Urban' did not match any bands. Available bands: [Percent_Tree_Cover, Percent_NonTree_Vegetation, Percent_NonVegetated, Quality, Percent_Tree_Cover_SD, Percent_NonVegetated_SD, Cloud]
Urban error 2005: reduce.mean: Error in map(ID=2005_03_06):
Image.select: Band pattern 'Urban' did not match any bands. Available bands: [Percent_Tree_Cover, Percent_NonTree_Vegetation, Percent_NonVegetated, Quality, Percent_Tree_Cover_SD, Percent_NonVegetated_SD, Cloud]
Urban error 2006: reduce.mean: Error in map(ID=2006_03_06):
Image.select: Band pattern 'Urban' did not match any bands. Available bands: [Percent_Tree_Cover, Percent_NonTree_Vegetation, Percent_NonVegetated, Quality, Percent_Tree_Cover_SD, Percent_NonVegetated_SD, Cloud]
Urban error 2007: reduce.mean: Error in map(ID=2007_03_06):
Image.select: Band pattern 'Urban' did not match any bands. Available bands: [Percent_Tree_Cover, Percent_NonTree_Vegetation

In [ ]:
import ee
import pandas as pd

ee.Initialize()

shanghai_geometry = ee.Geometry.Rectangle([120.85, 30.68, 122.12, 31.88])

years = list(range(2004, 2020))  

def get_urban_area(year):
    try:
        dataset = ee.Image(f'MODIS/006/MCD12Q1/{year}_01_01')  
        land_cover = dataset.select('LC_Type1')  
        urban_mask = land_cover.eq(13)
        urban_pixels = urban_mask.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=shanghai_geometry,
            scale=500,
            maxPixels=1e13
        ).get('LC_Type1')
        total_pixels = land_cover.reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=shanghai_geometry,
            scale=500,
            maxPixels=1e13
        ).get('LC_Type1')
        urban_percent = ee.Number(urban_pixels).divide(ee.Number(total_pixels)).multiply(100).getInfo()
        return {'year': year, 'urban_percent': urban_percent}
    except Exception as e:
        print(f"Urban error {year}: {e}")
        return {'year': year, 'urban_percent': None}
urban_data = [get_urban_area(year) for year in years]
df_urban = pd.DataFrame(urban_data)
df_urban.to_csv('C:/maps/datasets/shanghai_urban_2004_2019.csv', index=False)
print("Saved: C:/maps/datasets/shanghai_urban_2004_2019.csv")


Saved: C:/maps/datasets/shanghai_urban_2004_2019.csv


In [21]:
def get_no2_for_year(year):
    try:
        start = f'{year}-01-01'
        end = f'{year}-12-31'

        no2 = ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2') \
            .filterDate(start, end) \
            .select('NO2_column_number_density') \
            .mean()

        mean_dict = no2.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=shanghai_geometry,
            scale=1000,
            maxPixels=1e12
        )

        mean_no2 = mean_dict.get('NO2_column_number_density').getInfo()
        return {'year': year, 'no2_mean': mean_no2}
    except Exception as e:
        print(f"NO₂ error {year}: {e}")
        return {'year': year, 'no2_mean': None}

no2_years = list(range(2018, 2025))
no2_data = [get_no2_for_year(year) for year in no2_years]
df_no2 = pd.DataFrame(no2_data)
df_no2.to_csv('C:/maps/datasets/shanghai_no2_2018_2024.csv', index=False)
print("Saved NO₂ data to C:/maps/datasets/shanghai_no2_2018_2024.csv")


Saved NO₂ data to C:/maps/datasets/shanghai_no2_2018_2024.csv


In [22]:
import pandas as pd

# File paths
ndvi_path = "C:/maps/datasets/ndvi_shanghai_2004_2024.csv"
no2_path = "C:/maps/datasets/shanghai_no2_2018_2024.csv"
population_path = "C:/maps/datasets/shanghai_population_2000_2020_interpolated.csv"
rainfall_path = "C:/maps/datasets/shanghai_rainfall_2004_2024.csv"
temperature_path = "C:/maps/datasets/shanghai_temperature_2004_2024.csv"
urban_path = "C:/maps/datasets/shanghai_urban_2004_2019.csv"

# Load all datasets
ndvi = pd.read_csv(ndvi_path)
no2 = pd.read_csv(no2_path)
population = pd.read_csv(population_path)
rainfall = pd.read_csv(rainfall_path)
temperature = pd.read_csv(temperature_path)
urban = pd.read_csv(urban_path)

# Standardize column names
datasets = [ndvi, no2, population, rainfall, temperature, urban]
for df in datasets:
    if 'year' not in df.columns:
        df.columns = ['year'] + list(df.columns[1:])

# Merge all datasets on 'year'
combined = ndvi.merge(rainfall, on='year', how='outer') \
               .merge(temperature, on='year', how='outer') \
               .merge(population, on='year', how='outer') \
               .merge(urban, on='year', how='outer') \
               .merge(no2, on='year', how='outer')

# Sort and save
combined = combined.sort_values(by='year').reset_index(drop=True)
combined.to_csv("C:/maps/datasets/shanghai_combined_2004_2024.csv", index=False)
print("Saved to C:/maps/datasets/shanghai_combined_2004_2024.csv")


Saved to C:/maps/datasets/shanghai_combined_2004_2024.csv


In [ ]:
import pandas as pd
import numpy as np
import skfuzzy as fuzz
import matplotlib.pyplot as plt

# Load combined dataset
data = pd.read_csv("C:/maps/datasets/shanghai_combined_2004_2024.csv")
years = data['year']
X = data.drop(columns=['year']).values.T  
# Set number of clusters
n_clusters = 3

# Apply Fuzzy C-Means clustering
cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    X, c=n_clusters, m=2, error=0.005, maxiter=1000, init=None)
labels = np.argmax(u, axis=0)
data['cluster'] = labels
data['year'] = years

data.to_csv("C:/maps/datasets/shanghai_fcm_clustered_2004_2024.csv", index=False)
print("Clustering done and saved to shanghai_fcm_clustered_2004_2024.csv")




Clustering done and saved to shanghai_fcm_clustered_2004_2024.csv


In [24]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR
import matplotlib.pyplot as plt

# Load your clustered combined dataset
data_path = 'C:/maps/datasets/shanghai_combined_2004_2024.csv'
df = pd.read_csv(data_path)

# Ensure 'year' is set as index
df.set_index('year', inplace=True)

# Optional: display columns
print("Columns:", df.columns)

# ----- Step 1: ADF test -----
def adf_test(series, title=''):
    result = adfuller(series.dropna(), autolag='AIC')
    return result[1]  # p-value

# Find non-stationary columns
non_stationary_cols = []
for col in df.columns:
    p_val = adf_test(df[col])
    if p_val > 0.05:
        non_stationary_cols.append(col)
        print(f"❌ {col} not stationary (p={p_val:.4f})")
    else:
        print(f"✅ {col} is stationary (p={p_val:.4f})")

# ----- Step 2: Apply differencing if needed -----
df_diff = df.copy()
for col in non_stationary_cols:
    df_diff[col] = df[col].diff()

df_diff.dropna(inplace=True)

# ----- Step 3: Fit the VAR model -----
model = VAR(df_diff)
results = model.fit(maxlags=5, ic='aic')

# ----- Step 4: Forecast -----
forecast_steps = 21  # 2025 to 2045
forecast = results.forecast(df_diff.values[-results.k_ar:], steps=forecast_steps)

# Forecast is a NumPy array. Convert to DataFrame
forecast_index = list(range(2025, 2025 + forecast_steps))
forecast_df = pd.DataFrame(forecast, index=forecast_index, columns=df.columns)

# ----- Step 5: Inverse differencing (if needed) -----
df_forecast_final = forecast_df.copy()
for col in non_stationary_cols:
    last_value = df[col].iloc[-1]
    df_forecast_final[col] = forecast_df[col].cumsum() + last_value

# ----- Step 6: Extract NDVI prediction -----
ndvi_forecast = df_forecast_final[['ndvi']]
ndvi_forecast.to_csv('C:/maps/ndvi_forecast_2025_2045.csv')
print("✅ NDVI forecast saved to C:/maps/ndvi_forecast_2025_2045.csv")

# Optional: plot NDVI forecast
plt.figure(figsize=(10, 5))
plt.plot(df['ndvi'], label='Historical NDVI')
plt.plot(ndvi_forecast, label='Forecasted NDVI', linestyle='--')
plt.title("NDVI Forecast (2025–2045)")
plt.xlabel("Year")
plt.ylabel("NDVI")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


Columns: Index(['NDVI', 'rainfall', 'temperature', 'population', 'urban_percent',
       'no2_mean'],
      dtype='object')
❌ NDVI not stationary (p=0.8857)
❌ rainfall not stationary (p=0.0900)
❌ temperature not stationary (p=0.8621)
❌ population not stationary (p=0.9481)
❌ urban_percent not stationary (p=0.9910)
✅ no2_mean is stationary (p=0.0000)


C:\Users\irctc\AppData\Roaming\Python\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)


ValueError: maxlags is too large for the number of observations and the number of equations. The largest model cannot be estimated.

In [25]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR
import matplotlib.pyplot as plt

# Load your clustered combined dataset
data_path = 'C:/maps/datasets/shanghai_combined_2004_2024.csv'
df = pd.read_csv(data_path)

# Ensure 'year' is set as index
df.set_index('year', inplace=True)

# Optional: display columns
print("Columns:", df.columns)

# ----- Step 1: ADF test -----
def adf_test(series, title=''):
    result = adfuller(series.dropna(), autolag='AIC')
    return result[1]  # p-value

# Find non-stationary columns
non_stationary_cols = []
for col in df.columns:
    p_val = adf_test(df[col])
    if p_val > 0.05:
        non_stationary_cols.append(col)
        print(f"❌ {col} not stationary (p={p_val:.4f})")
    else:
        print(f"✅ {col} is stationary (p={p_val:.4f})")

# ----- Step 2: Apply differencing if needed -----
df_diff = df.copy()
for col in non_stationary_cols:
    df_diff[col] = df[col].diff()

df_diff.dropna(inplace=True)

# ----- Step 3: Fit the VAR model -----
model = VAR(df_diff)
results = model.fit(maxlags=5, ic='aic')

# ----- Step 4: Forecast -----
forecast_steps = 21  # 2025 to 2045
forecast = results.forecast(df_diff.values[-results.k_ar:], steps=forecast_steps)

# Forecast is a NumPy array. Convert to DataFrame
forecast_index = list(range(2025, 2025 + forecast_steps))
forecast_df = pd.DataFrame(forecast, index=forecast_index, columns=df.columns)

# ----- Step 5: Inverse differencing (if needed) -----
df_forecast_final = forecast_df.copy()
for col in non_stationary_cols:
    last_value = df[col].iloc[-1]
    df_forecast_final[col] = forecast_df[col].cumsum() + last_value

# ----- Step 6: Extract NDVI prediction -----
ndvi_forecast = df_forecast_final[['ndvi']]
ndvi_forecast.to_csv('C:/maps/ndvi_forecast_2025_2045.csv')
print("✅ NDVI forecast saved to C:/maps/ndvi_forecast_2025_2045.csv")

Columns: Index(['NDVI', 'rainfall', 'temperature', 'population', 'urban_percent',
       'no2_mean'],
      dtype='object')
❌ NDVI not stationary (p=0.8857)
❌ rainfall not stationary (p=0.0900)
❌ temperature not stationary (p=0.8621)
❌ population not stationary (p=0.9481)
❌ urban_percent not stationary (p=0.9910)
✅ no2_mean is stationary (p=0.0000)


C:\Users\irctc\AppData\Roaming\Python\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)


ValueError: maxlags is too large for the number of observations and the number of equations. The largest model cannot be estimated.

In [26]:
import pandas as pd
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller

# Load combined dataset
df = pd.read_csv("C:/maps/datasets/shanghai_combined_2004_2024.csv")

# Set datetime index (important for forecasting)
df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')
df.set_index('year', inplace=True)

# ADF Test for stationarity
def check_stationarity(series):
    result = adfuller(series)
    return result[1] < 0.05  # True if stationary

non_stationary_cols = []
for col in df.columns:
    if not check_stationarity(df[col]):
        print(f"❌ {col} not stationary (p={adfuller(df[col])[1]:.4f})")
        non_stationary_cols.append(col)
    else:
        print(f"✅ {col} is stationary (p={adfuller(df[col])[1]:.4f})")

# Differencing non-stationary columns
df_diff = df.copy()
for col in non_stationary_cols:
    df_diff[col] = df[col].diff()

df_diff.dropna(inplace=True)

# Train VAR model
model = VAR(df_diff)
results = model.fit(ic='aic')  # Automatically select best lag using AIC

# Forecast future steps (2025 to 2045 => 21 years)
forecast_steps = 21
forecast = results.forecast(df_diff.values[-results.k_ar:], steps=forecast_steps)

# Create forecast DataFrame
forecast_index = pd.date_range(start='2025', periods=forecast_steps, freq='Y')
forecast_df = pd.DataFrame(forecast, columns=df.columns, index=forecast_index)

# Inverse differencing for NDVI to get actual predicted values
ndvi_last_actual = df['NDVI'].iloc[-1]
forecast_df['NDVI'] = forecast_df['NDVI'].cumsum() + ndvi_last_actual

# Save predicted NDVI
forecast_df[['NDVI']].to_csv("C:/maps/datasets/shanghai_predicted_ndvi_2025_2045.csv")
print("✅ Saved NDVI forecast to C:/maps/datasets/shanghai_predicted_ndvi_2025_2045.csv")


C:\Users\irctc\AppData\Local\Temp\ipykernel_20484\1986325655.py:9: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')


MissingDataError: exog contains inf or nans

In [28]:
import pandas as pd
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
import numpy as np

# Load dataset
df = pd.read_csv("C:/maps/datasets/shanghai_combined_2004_2024.csv")

# Set datetime index
df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')
df.set_index('year', inplace=True)

# Remove inf/nan
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Function: Check stationarity with error handling
def check_stationarity(series):
    if series.isnull().any() or len(series) < 10:
        return False  # Not enough data for ADF
    try:
        return adfuller(series)[1] < 0.05
    except ValueError:
        return False

# Identify non-stationary columns
non_stationary_cols = []
for col in df.columns:
    is_stationary = check_stationarity(df[col])
    p_value = adfuller(df[col])[1] if is_stationary else None
    if not is_stationary:
        print(f"❌ {col} not stationary")
        non_stationary_cols.append(col)
    else:
        print(f"✅ {col} is stationary (p={p_value:.4f})")

# Apply differencing to non-stationary columns
df_diff = df.copy()
for col in non_stationary_cols:
    df_diff[col] = df[col].diff()

# Drop rows with NaN (from differencing)
df_diff.dropna(inplace=True)

# Fit VAR model
model = VAR(df_diff)
results = model.fit(ic='aic')

# Forecast NDVI for 2025–2045
forecast_steps = 21
forecast = results.forecast(df_diff.values[-results.k_ar:], steps=forecast_steps)

# Prepare forecast DataFrame
forecast_index = pd.date_range(start='2025', periods=forecast_steps, freq='Y')
forecast_df = pd.DataFrame(forecast, columns=df.columns, index=forecast_index)

# Inverse differencing for NDVI (only if differenced)
if 'NDVI' in non_stationary_cols:
    last_ndvi = df['NDVI'].iloc[-1]
    forecast_df['NDVI'] = forecast_df['NDVI'].cumsum() + last_ndvi

# Save only NDVI forecast
forecast_df[['NDVI']].to_csv("C:/maps/datasets/shanghai_predicted_ndvi_2025_2045.csv")
print("✅ Forecast saved to C:/maps/datasets/shanghai_predicted_ndvi_2025_2045.csv")



❌ NDVI not stationary
❌ rainfall not stationary
❌ temperature not stationary
❌ population not stationary
❌ urban_percent not stationary
❌ no2_mean not stationary


C:\Users\irctc\AppData\Local\Temp\ipykernel_20484\4213329227.py:10: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')
C:\Users\irctc\AppData\Roaming\Python\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


KeyError: 'aic'

In [29]:
import pandas as pd
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
import numpy as np

# Load dataset
df = pd.read_csv("C:/maps/datasets/shanghai_combined_2004_2024.csv")

# Set datetime index
df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')
df.set_index('year', inplace=True)

# Remove inf/nan
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Function: Check stationarity with error handling
def check_stationarity(series):
    if series.isnull().any() or len(series) < 10:
        return False  # Not enough data for ADF
    try:
        return adfuller(series)[1] < 0.05
    except ValueError:
        return False

# Identify non-stationary columns
non_stationary_cols = []
for col in df.columns:
    is_stationary = check_stationarity(df[col])
    p_value = adfuller(df[col])[1] if is_stationary else None
    if not is_stationary:
        print(f"❌ {col} not stationary")
        non_stationary_cols.append(col)
    else:
        print(f"✅ {col} is stationary (p={p_value:.4f})")

# Apply differencing to non-stationary columns
df_diff = df.copy()
for col in non_stationary_cols:
    df_diff[col] = df[col].diff()

# Drop rows with NaN (from differencing)
df_diff.dropna(inplace=True)

# Fit VAR model
model = VAR(df_diff)
results = model.fit(ic='aic')

# Forecast NDVI for 2025–2045
forecast_steps = 21
forecast = results.forecast(df_diff.values[-results.k_ar:], steps=forecast_steps)

# Prepare forecast DataFrame
forecast_index = pd.date_range(start='2025', periods=forecast_steps, freq='Y')
forecast_df = pd.DataFrame(forecast, columns=df.columns, index=forecast_index)

# Inverse differencing for NDVI (only if differenced)
if 'NDVI' in non_stationary_cols:
    last_ndvi = df['NDVI'].iloc[-1]
    forecast_df['NDVI'] = forecast_df['NDVI'].cumsum() + last_ndvi

# Save only NDVI forecast
forecast_df[['NDVI']].to_csv("C:/maps/datasets/shanghai_predicted_ndvi_2025_2045.csv")
print("✅ Forecast saved to C:/maps/datasets/shanghai_predicted_ndvi_2025_2045.csv")


❌ NDVI not stationary
❌ rainfall not stationary
❌ temperature not stationary
❌ population not stationary
❌ urban_percent not stationary
❌ no2_mean not stationary


C:\Users\irctc\AppData\Local\Temp\ipykernel_20484\381680787.py:10: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')
C:\Users\irctc\AppData\Roaming\Python\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


KeyError: 'aic'

In [30]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller

# Load and prepare dataset
df = pd.read_csv("C:/maps/datasets/shanghai_combined_2004_2024.csv")
df['year'] = pd.date_range(start='2004', periods=len(df), freq='YE')
df.set_index('year', inplace=True)

# Remove inf/nan
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# ADF Test
def check_stationarity(series):
    try:
        return adfuller(series.dropna())[1] < 0.05
    except:
        return False

# Track non-stationary columns
non_stationary_cols = []
for col in df.columns:
    if not check_stationarity(df[col]):
        print(f"❌ {col} not stationary")
        non_stationary_cols.append(col)
    else:
        print(f"✅ {col} is stationary")

# Apply differencing
df_diff = df.copy()
for col in non_stationary_cols:
    df_diff[col] = df[col].diff()

df_diff.dropna(inplace=True)

# Fit VAR with manually selected lag using AIC
model = VAR(df_diff)
lag_order = model.select_order(maxlags=5)

# Check available criteria
print("Lag Order Selection:")
print(lag_order.summary())

# Use best lag from AIC (fallback to 1 if None)
selected_lag = lag_order.aic
if selected_lag is None:
    print("⚠️ AIC not available, using lag=1")
    selected_lag = 1

results = model.fit(selected_lag)

# Forecast
forecast_steps = 21  # 2025 to 2045
forecast = results.forecast(df_diff.values[-selected_lag:], steps=forecast_steps)
forecast_index = pd.date_range(start='2025', periods=forecast_steps, freq='YE')
forecast_df = pd.DataFrame(forecast, columns=df.columns, index=forecast_index)

# Inverse transform for NDVI if differenced
if 'NDVI' in non_stationary_cols:
    forecast_df['NDVI'] = forecast_df['NDVI'].cumsum() + df['NDVI'].iloc[-1]

# Save only NDVI forecast
forecast_df[['NDVI']].to_csv("C:/maps/datasets/shanghai_predicted_ndvi_2025_2045.csv")
print("✅ Forecast saved to C:/maps/datasets/shanghai_predicted_ndvi_2025_2045.csv")


❌ NDVI not stationary
❌ rainfall not stationary
❌ temperature not stationary
❌ population not stationary
❌ urban_percent not stationary
❌ no2_mean not stationary


C:\Users\irctc\AppData\Roaming\Python\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


ValueError: maxlags is too large for the number of observations and the number of equations. The largest model cannot be estimated.

In [31]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR
import matplotlib.pyplot as plt

# ----- Step 1: Load and preprocess the data -----
file_path = "C:/maps/datasets/shanghai_combined_2004_2024.csv"
df = pd.read_csv(file_path)

# Set datetime index
df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')
df.set_index('year', inplace=True)

# Check and drop NaNs
df = df.dropna()

# ----- Step 2: Check stationarity and difference if needed -----
def check_stationarity(series):
    result = adfuller(series.dropna())
    return result[1] < 0.05  # p-value < 0.05 means stationary

non_stationary_cols = []
for col in df.columns:
    if not check_stationarity(df[col]):
        print(f"❌ {col} not stationary (p={adfuller(df[col])[1]:.4f})")
        non_stationary_cols.append(col)
    else:
        print(f"✅ {col} is stationary (p={adfuller(df[col])[1]:.4f})")

# Difference non-stationary columns
df_diff = df.copy()
df_diff[non_stationary_cols] = df_diff[non_stationary_cols].diff().dropna()

# Drop first row after differencing
df_diff = df_diff.dropna()

# ----- Step 3: Fit VAR model with dynamic maxlags -----
model = VAR(df_diff)

# Try lags from 5 down to 1
for max_lag in range(5, 0, -1):
    try:
        lag_order = model.select_order(maxlags=max_lag)
        selected_lag = lag_order.aic or lag_order.bic or 1
        print(f"✅ Selected lag: {selected_lag}")
        break
    except ValueError as e:
        print(f"⚠️ maxlags={max_lag} failed: {e}")
        continue
else:
    raise Exception("❌ Could not find valid lag order.")

# Fit model
results = model.fit(selected_lag)

# ----- Step 4: Forecast 21 years ahead (2025 to 2045) -----
forecast_steps = 21
forecast = results.forecast(df_diff.values[-selected_lag:], steps=forecast_steps)

# Convert to DataFrame
forecast_df = pd.DataFrame(forecast, columns=df.columns)
forecast_df.index = pd.date_range(start='2025', periods=forecast_steps, freq='Y')

# ----- Step 5: Reconstruct original NDVI from differenced -----
last_known = df.iloc[-1]
forecast_cumsum = forecast_df.cumsum()
forecast_reconstructed = forecast_cumsum.add(last_known)

# ----- Step 6: Plot NDVI forecast -----
plt.figure(figsize=(10, 5))
plt.plot(df.index, df['NDVI'], label='Historical NDVI')
plt.plot(forecast_reconstructed.index, forecast_reconstructed['NDVI'], label='Forecasted NDVI (2025-2045)', linestyle='--')
plt.xlabel('Year')
plt.ylabel('NDVI')
plt.title('NDVI Forecast for Shanghai (2004–2045)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# ----- Step 7: Save forecasted NDVI -----
forecast_reconstructed[['NDVI']].to_csv("C:/maps/ndvi_forecast_2025_2045.csv")
print("✅ Forecast saved to: C:/maps/ndvi_forecast_2025_2045.csv")


C:\Users\irctc\AppData\Local\Temp\ipykernel_20484\844066718.py:12: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')


ValueError: sample size is too short to use selected regression component

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR

# Load data
file_path = "C:/maps/datasets/shanghai_combined_2004_2024.csv"
df = pd.read_csv(file_path)

# Set datetime index
df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')
df.set_index('year', inplace=True)

# Drop any NaN or Inf just in case
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Check for stationarity
def check_stationarity(series):
    result = adfuller(series.dropna())
    return result[1] < 0.05

non_stationary_cols = []
for col in df.columns:
    if not check_stationarity(df[col]):
        print(f"❌ {col} not stationary (p={adfuller(df[col])[1]:.4f})")
        non_stationary_cols.append(col)
    else:
        print(f"✅ {col} is stationary (p={adfuller(df[col])[1]:.4f})")

# Difference non-stationary columns
df_diff = df.copy()
df_diff[non_stationary_cols] = df_diff[non_stationary_cols].diff()
df_diff.dropna(inplace=True)

# Fit VAR model
model = VAR(df_diff)

# Use auto lag selection safely
try:
    lag_order = model.select_order(maxlags=5)
    selected_lag = lag_order.selected_orders.get('aic') or 1
    print(f"✅ Selected lag order: {selected_lag}")
except Exception as e:
    print("⚠️ Lag order selection failed, defaulting to lag=1")
    selected_lag = 1

results = model.fit(selected_lag)

# Forecast 21 years (2025–2045)
forecast_steps = 21
forecast = results.forecast(df_diff.values[-selected_lag:], steps=forecast_steps)

# Create DataFrame for forecast
forecast_df = pd.DataFrame(forecast, columns=df.columns)
forecast_df.index = pd.date_range(start='2025', periods=forecast_steps, freq='Y')

# Reconstruct original scale from differenced
last_known = df.iloc[-1]
forecast_reconstructed = forecast_df.cumsum() + last_known

# Plot historical and forecasted NDVI
plt.figure(figsize=(10, 5))
plt.plot(df.index, df['NDVI'], label='Historical NDVI')
plt.plot(forecast_reconstructed.index, forecast_reconstructed['NDVI'],
         label='Forecasted NDVI (2025–2045)', linestyle='--')
plt.xlabel('Year')
plt.ylabel('NDVI')
plt.title('NDVI Forecast for Shanghai (2004–2045)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Save forecast to CSV
forecast_reconstructed[['NDVI']].to_csv("C:/maps/ndvi_forecast_2025_2045.csv")
print("✅ Forecast saved: C:/maps/ndvi_forecast_2025_2045.csv")


C:\Users\irctc\AppData\Local\Temp\ipykernel_20484\1117315787.py:12: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')


ValueError: sample size is too short to use selected regression component

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR

# Load your dataset
file_path = "C:/maps/datasets/shanghai_combined_2004_2024.csv"
df = pd.read_csv(file_path)

# Add datetime index
df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')
df.set_index('year', inplace=True)

# Replace problematic values and drop NaNs
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# ADF test with sample size check
def check_stationarity(series):
    series = series.dropna()
    if len(series) < 10:  # Arbitrary threshold to avoid ADF crash
        print(f"⚠️ Skipping ADF test for short series ({series.name})")
        return False
    result = adfuller(series)
    return result[1] < 0.05

# Check all columns
non_stationary_cols = []
for col in df.columns:
    try:
        if not check_stationarity(df[col]):
            print(f"❌ {col} not stationary (p={adfuller(df[col].dropna())[1]:.4f})")
            non_stationary_cols.append(col)
        else:
            print(f"✅ {col} is stationary (p={adfuller(df[col].dropna())[1]:.4f})")
    except Exception as e:
        print(f"⚠️ Skipping {col} due to error: {e}")
        non_stationary_cols.append(col)

# Apply first difference to non-stationary columns
df_diff = df.copy()
df_diff[non_stationary_cols] = df_diff[non_stationary_cols].diff()
df_diff.dropna(inplace=True)

# Fit VAR model
model = VAR(df_diff)

# Auto lag selection with fallback
try:
    lag_order = model.select_order(maxlags=5)
    selected_lag = lag_order.selected_orders.get('aic') or 1
    print(f"✅ Selected lag order: {selected_lag}")
except Exception as e:
    print(f"⚠️ Lag selection failed: {e}, defaulting to lag=1")
    selected_lag = 1

results = model.fit(selected_lag)

# Forecast 21 steps (2025–2045)
forecast_steps = 21
forecast = results.forecast(df_diff.values[-selected_lag:], steps=forecast_steps)

# Rebuild to original scale
forecast_df = pd.DataFrame(forecast, columns=df.columns)
forecast_df.index = pd.date_range(start='2025', periods=forecast_steps, freq='Y')
last_known = df.iloc[-1]
forecast_reconstructed = forecast_df.cumsum() + last_known

# Plot
plt.figure(figsize=(10, 5))
plt.plot(df.index, df['NDVI'], label='Historical NDVI')
plt.plot(forecast_reconstructed.index, forecast_reconstructed['NDVI'],
         label='Forecasted NDVI (2025–2045)', linestyle='--')
plt.xlabel('Year')
plt.ylabel('NDVI')
plt.title('NDVI Forecast for Shanghai (2004–2045)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Save to CSV
forecast_reconstructed[['NDVI']].to_csv("C:/maps/ndvi_forecast_2025_2045.csv")
print("✅ Forecast saved at C:/maps/ndvi_forecast_2025_2045.csv")


C:\Users\irctc\AppData\Local\Temp\ipykernel_20484\3011998416.py:12: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')
C:\Users\irctc\AppData\Roaming\Python\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


⚠️ Skipping ADF test for short series (NDVI)
⚠️ Skipping NDVI due to error: sample size is too short to use selected regression component
⚠️ Skipping ADF test for short series (rainfall)
⚠️ Skipping rainfall due to error: sample size is too short to use selected regression component
⚠️ Skipping ADF test for short series (temperature)
⚠️ Skipping temperature due to error: sample size is too short to use selected regression component
⚠️ Skipping ADF test for short series (population)
⚠️ Skipping population due to error: sample size is too short to use selected regression component
⚠️ Skipping ADF test for short series (urban_percent)
⚠️ Skipping urban_percent due to error: sample size is too short to use selected regression component
⚠️ Skipping ADF test for short series (no2_mean)
⚠️ Skipping no2_mean due to error: sample size is too short to use selected regression component
⚠️ Lag selection failed: maxlags is too large for the number of observations and the number of equations. The la

ValueError: zero-size array to reduction operation maximum which has no identity

In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR

# Load your dataset
df = pd.read_csv("C:/maps/datasets/shanghai_combined_2004_2024.csv")

# Convert 'year' to datetime index
df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')
df.set_index('year', inplace=True)

# Replace inf and drop NaNs
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Differencing to make it stationary
df_diff = df.diff().dropna()

# Fit VAR model
model = VAR(df_diff)
lag_order = model.select_order(maxlags=5).selected_orders['aic']
results = model.fit(lag_order)

# Forecast for next 21 years (2025 to 2045)
forecast_steps = 21
forecast = results.forecast(df_diff.values[-lag_order:], steps=forecast_steps)

# Convert forecast to DataFrame
forecast_df = pd.DataFrame(forecast, columns=df.columns)
forecast_df.index = pd.date_range(start='2025', periods=forecast_steps, freq='Y')

# Reconstruct actual values from differencing
last_known = df.iloc[-1]
forecast_reconstructed = forecast_df.cumsum() + last_known

# Extract only NDVI
ndvi_forecast = forecast_reconstructed[['NDVI']]

# Plot
plt.figure(figsize=(10, 5))
plt.plot(df.index, df['NDVI'], label='Historical NDVI')
plt.plot(ndvi_forecast.index, ndvi_forecast['NDVI'], label='Forecasted NDVI', linestyle='--')
plt.xlabel('Year')
plt.ylabel('NDVI')
plt.title('NDVI Forecast for Shanghai (2004–2045)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Save forecast
ndvi_forecast.to_csv("C:/maps/datasets/ndvi_forecast_2025_2045.csv")


C:\Users\irctc\AppData\Local\Temp\ipykernel_20484\3401903624.py:10: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  df['year'] = pd.date_range(start='2004', periods=len(df), freq='Y')
C:\Users\irctc\AppData\Roaming\Python\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


ValueError: maxlags is too large for the number of observations and the number of equations. The largest model cannot be estimated.

In [35]:
# Install Prophet if needed
!pip install prophet

# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
import os

# Load the original dataset
file_path = '/mnt/data/shanghai_fcm_clustered_2004_2024.csv'
df = pd.read_csv(file_path)

# Preview the data
print("Original Dataset Preview:")
print(df.head())

# Rename columns (change these if your actual column names are different)
df.rename(columns={'Year': 'ds', 'NDVI': 'y'}, inplace=True)

# Convert year to datetime
df['ds'] = pd.to_datetime(df['ds'], format='%Y')

# Fit the Prophet model
model = Prophet()
model.fit(df)

# Forecast for the next 20 years
future = model.make_future_dataframe(periods=20, freq='Y')
forecast = model.predict(future)

# Filter only future predictions (2025–2044)
predicted = forecast[forecast['ds'].dt.year > 2024].copy()

# Create the final prediction dataset
predicted_dataset = predicted[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
predicted_dataset['Year'] = predicted_dataset['ds'].dt.year
predicted_dataset.rename(columns={
    'yhat': 'Predicted_NDVI',
    'yhat_lower': 'Lower_Bound',
    'yhat_upper': 'Upper_Bound'
}, inplace=True)

# Drop datetime column and rearrange
predicted_dataset = predicted_dataset[['Year', 'Predicted_NDVI', 'Lower_Bound', 'Upper_Bound']]

# Define save path
save_path = r'C:\maps\datasets\ndvi_predictions_2025_2044.csv'

# Ensure directory exists
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# Save the new dataset
predicted_dataset.to_csv(save_path, index=False)

print(f"\n✅ Predicted NDVI values saved to:\n{save_path}")
print("\n📈 Here's a preview:")
print(predicted_dataset.head())


Defaulting to user installation because normal site-packages is not writeable
  Using cached prophet-1.1.6-py3-none-win_amd64.whl.metadata (3.6 kB)
  Using cached cmdstanpy-1.2.5-py3-none-any.whl.metadata (4.0 kB)
  Using cached holidays-0.69-py3-none-any.whl.metadata (28 kB)
Using cached prophet-1.1.6-py3-none-win_amd64.whl (13.3 MB)
Using cached cmdstanpy-1.2.5-py3-none-any.whl (94 kB)
Using cached holidays-0.69-py3-none-any.whl (863 kB)


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/shanghai_fcm_clustered_2004_2024.csv'

In [36]:


# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
import os

# ✅ Update the local file path to your actual CSV
file_path = r'C:\maps\datasets\shanghai_fcm_clustered_2004_2024.csv'

# Load the dataset
df = pd.read_csv(file_path)

# Preview the data
print("Original Dataset Preview:")
print(df.head())

# Rename columns if needed
df.rename(columns={'Year': 'ds', 'NDVI': 'y'}, inplace=True)

# Convert year to datetime
df['ds'] = pd.to_datetime(df['ds'], format='%Y')

# Train Prophet model
model = Prophet()
model.fit(df)

# Forecast 20 future years
future = model.make_future_dataframe(periods=20, freq='Y')
forecast = model.predict(future)

# Filter for years after 2024
predicted = forecast[forecast['ds'].dt.year > 2024].copy()

# Create a new dataset for predicted NDVI
predicted_dataset = predicted[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
predicted_dataset['Year'] = predicted_dataset['ds'].dt.year
predicted_dataset.rename(columns={
    'yhat': 'Predicted_NDVI',
    'yhat_lower': 'Lower_Bound',
    'yhat_upper': 'Upper_Bound'
}, inplace=True)

# Reorder and clean
predicted_dataset = predicted_dataset[['Year', 'Predicted_NDVI', 'Lower_Bound', 'Upper_Bound']]

# ✅ Save the predicted dataset locally
save_path = r'C:\maps\datasets\ndvi_predictions_2025_2044.csv'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
predicted_dataset.to_csv(save_path, index=False)

# Show result
print(f"\n✅ Predicted NDVI values saved to: {save_path}")
print("\n📈 Preview:")
print(predicted_dataset.head())


Original Dataset Preview:
   year      NDVI    rainfall  temperature  population  urban_percent  \
0  2000       NaN         NaN          NaN  17000000.0            NaN   
1  2001       NaN         NaN          NaN  17300000.0            NaN   
2  2002       NaN         NaN          NaN  17600000.0            NaN   
3  2003       NaN         NaN          NaN  17900000.0            NaN   
4  2004  0.364792  954.587438    16.793247  18200000.0       21.97004   

   no2_mean  cluster  
0       NaN        0  
1       NaN        0  
2       NaN        0  
3       NaN        0  
4       NaN        0  


KeyError: 'ds'

In [1]:



import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
import os
file_path = r'C:\maps\datasets\shanghai_fcm_clustered_2004_2024.csv'
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip().str.lower()
df.rename(columns={'year': 'ds', 'ndvi': 'y'}, inplace=True)
df['ds'] = pd.to_datetime(df['ds'], format='%Y')
model = Prophet()
model.fit(df)
future = model.make_future_dataframe(periods=20, freq='Y')
forecast = model.predict(future)
predicted = forecast[forecast['ds'].dt.year > 2024].copy()
predicted_dataset = predicted[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
predicted_dataset['Year'] = predicted_dataset['ds'].dt.year
predicted_dataset.rename(columns={
    'yhat': 'Predicted_NDVI',
    'yhat_lower': 'Lower_Bound',
    'yhat_upper': 'Upper_Bound'
}, inplace=True)
predicted_dataset = predicted_dataset[['Year', 'Predicted_NDVI', 'Lower_Bound', 'Upper_Bound']]
save_path = r'C:\maps\datasets\ndvi_predictions_2025_2044.csv'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
predicted_dataset.to_csv(save_path, index=False)
print(f"\n✅ Predicted NDVI values saved to: {save_path}")
print("\n📈 Preview:")
print(predicted_dataset.head())


10:33:21 - cmdstanpy - INFO - Chain [1] start processing
10:33:22 - cmdstanpy - INFO - Chain [1] done processing



✅ Predicted NDVI values saved to: C:\maps\datasets\ndvi_predictions_2025_2044.csv

📈 Preview:
    Year  Predicted_NDVI  Lower_Bound  Upper_Bound
26  2025        0.384929     0.375490     0.393954
27  2026        0.378718     0.369535     0.388045
28  2027        0.370486     0.361221     0.380447
29  2028        0.400160     0.390393     0.409843
30  2029        0.395985     0.386912     0.406450


C:\Users\irctc\AppData\Roaming\Python\Python312\site-packages\prophet\forecaster.py:1854: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  dates = pd.date_range(
